In [1]:
# 의존성 설치 (transformers v5는 bitsandbytes 이슈 있어 <5 고정)
!pip install -U "transformers>=4.44,<5" "peft>=0.11" "bitsandbytes>=0.46.1" "accelerate>=0.30" datasets sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 393.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 173.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 128.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 267.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 240.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.9/799.9 kB 138.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 157.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 176.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 266.5 MB/s eta 0:00:00

[notice] A new release of pip

In [2]:
# 설정 & import
import os, torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer, DataCollatorForSeq2Seq)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL      = "kakaocorp/kanana-1.5-8b-instruct-2505"  # 공공 적합 한국어 모델
TRAIN_FILE = "kanana_train.jsonl"
VALID_FILE = "kanana_valid.jsonl"
OUT_DIR    = "/workspace/kanana_workit_out"
MAX_LEN    = 8192   # RPT 최대 6593 토큰 커버 (A40 48GB에서 OK)
SEED       = 42

In [3]:
# 토크나이저 + 데이터 전처리
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token   # 패딩 토큰 없으면 eos로

# messages(system/user/assistant) → input_ids + labels
# 핵심: 프롬프트(system+user) 구간은 -100으로 마스킹 → assistant 응답만 loss 계산
def build(ex):
    m = ex["messages"]
    prompt = tok.apply_chat_template(m[:-1], add_generation_prompt=True,  tokenize=True)  # assistant 직전까지
    full   = tok.apply_chat_template(m,      add_generation_prompt=False, tokenize=True)  # 전체
    labels = [-100]*len(prompt) + full[len(prompt):]      # 프롬프트 구간 마스킹
    full, labels = full[:MAX_LEN], labels[:MAX_LEN]
    return {"input_ids": full, "attention_mask": [1]*len(full), "labels": labels}

train_ds = load_dataset("json", data_files=TRAIN_FILE, split="train").shuffle(seed=SEED)
valid_ds = load_dataset("json", data_files=VALID_FILE, split="train")
cols = train_ds.column_names
train_ds = train_ds.map(build, remove_columns=cols)
valid_ds = valid_ds.map(build, remove_columns=cols)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/405 [00:00<?, ? examples/s]

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

In [4]:
# 길이 확인(토큰체크)
# 8192 초과 건수가 0이어야 정답(assistant)이 안 잘림
mx  = max(len(x["input_ids"]) for x in train_ds)
ovr = sum(len(x["input_ids"]) >= MAX_LEN for x in train_ds)
print(f"train {len(train_ds)} / valid {len(valid_ds)} / 최대 {mx}토큰 / {MAX_LEN}초과 {ovr}건")

train 405 / valid 45 / 최대 6593토큰 / 8192초과 0건


In [5]:
# 4bit 모델 로드 + LoRA 부착
# QLoRA: 4bit nf4 양자화로 8B를 A40 한 장에 로드
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
        device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False   # gradient checkpointing과 충돌 방지

# LoRA 어댑터 (r16/α32, 어텐션+MLP 7개 모듈)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # 학습 파라미터 비율 확인

config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 41,943,040 || all params: 8,072,228,864 || trainable%: 0.5196


In [6]:
# collator + 학습 인자
# 동적 패딩(배치 내 최장에 맞춤) → 짧은 배치는 짧게, 긴 RPT 배치만 길어짐
collator = DataCollatorForSeq2Seq(tok, model=model, label_pad_token_id=-100, padding="longest")

args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=3,
    save_strategy="epoch",              # epoch마다 어댑터 저장
    eval_strategy="epoch",              # epoch마다 valid loss (구버전이면 evaluation_strategy)
    save_total_limit=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,      # 유효 batch = 1×8 = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    optim="paged_adamw_8bit",           # 메모리 절약 옵티마이저
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5,
    report_to="none",
    seed=SEED,
)

In [7]:
# 학습 실행
trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=valid_ds,
                  data_collator=collator)
trainer.train()   # epoch별로 train/valid loss 로그가 찍힘

/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


Epoch,Training Loss,Validation Loss
1,0.536300,0.494360
2,0.271400,0.439060
3,0.137200,0.479390


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


TrainOutput(global_step=153, training_loss=0.3628503970460954, metrics={'train_runtime': 3779.6156, 'train_samples_per_second': 0.321, 'train_steps_per_second': 0.04, 'total_flos': 1.5353831522304e+17, 'train_loss': 0.3628503970460954, 'epoch': 3.0})

In [8]:
# 최종 저장
trainer.save_model(os.path.join(OUT_DIR, "final"))   # 마지막 어댑터
tok.save_pretrained(os.path.join(OUT_DIR, "final"))
print("완료 → checkpoint-*(epoch별) + final")
print("valid loss 최저 epoch의 checkpoint를 골라 추론에 사용")

완료 → checkpoint-*(epoch별) + final
valid loss 최저 epoch의 checkpoint를 골라 추론에 사용


In [ ]:
import json, torch, re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm

# ── 설정 ──
MODEL   = "kakaocorp/kanana-1.5-8b-instruct-2505"
ADAPTER = "/workspace/kanana_workit_out/checkpoint-153"   # ← epoch2 체크포인트 경로로 수정!
VALID   = "kanana_valid.jsonl"

# ── 모델 로드 (학습과 동일한 4bit) ──
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
        device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True)
model = PeftModel.from_pretrained(base, ADAPTER).eval()

# ── 출력에서 판정 파싱 ──
def parse_판정(text):
    try:
        d = json.loads(text)
        return d.get("판정") or d.get("label")
    except:
        m = re.search(r'(판정|label)"?\s*[:：]\s*"?(충족|불가|검토|일치|불일치)', text)
        return m.group(2) if m else "PARSE_FAIL"

# ── valid 전체 추론 ──
gold, pred, tasks = [], [], []
for line in tqdm(list(open(VALID, encoding='utf-8'))):   # ← tqdm으로 감쌈
    rec = json.loads(line)
    g = rec["meta"].get("판정") or rec["meta"].get("label")     # 정답
    enc = tok.apply_chat_template(rec["messages"][:-1], add_generation_prompt=True,
                                  return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(enc, max_new_tokens=512, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    ans = tok.decode(out[0][enc.shape[1]:], skip_special_tokens=True)
    gold.append(g)
    pred.append(parse_판정(ans))
    tasks.append(rec["meta"].get("task"))

# ── 순수 파이썬 지표 (sklearn 불필요) ──
def evaluate(gold, pred):
    labels = sorted(set(gold) | set(pred))
    n = len(gold)
    acc = sum(g == p for g, p in zip(gold, pred)) / n if n else 0
    print(f"{'label':10}{'prec':>7}{'rec':>7}{'f1':>7}{'n':>5}")
    f1s = []
    for L in labels:
        tp = sum(g == L and p == L for g, p in zip(gold, pred))
        fp = sum(g != L and p == L for g, p in zip(gold, pred))
        fn = sum(g == L and p != L for g, p in zip(gold, pred))
        prec = tp/(tp+fp) if tp+fp else 0
        rec  = tp/(tp+fn) if tp+fn else 0
        f1   = 2*prec*rec/(prec+rec) if prec+rec else 0
        support = sum(g == L for g in gold)
        if support > 0: f1s.append(f1)          # macro는 정답에 존재하는 클래스만 평균
        print(f"{L:10}{prec:7.3f}{rec:7.3f}{f1:7.3f}{support:5}")
    macro = sum(f1s)/len(f1s) if f1s else 0
    print(f"\naccuracy : {acc:.3f}")
    print(f"macro-F1 : {macro:.3f}")
    print(f"파싱실패 : {pred.count('PARSE_FAIL')}")
    print("혼동행렬 (행=정답 / 열=예측)")
    print("        " + "".join(f"{p:>8}" for p in labels))
    for gg in labels:
        row = [sum(a == gg and b == p for a, b in zip(gold, pred)) for p in labels]
        print(f"{gg:>8}" + "".join(f"{c:>8}" for c in row))

# ── 태스크별 + 전체 ──
for T in ["CON", "PEP", "RPT"]:
    idx = [i for i, t in enumerate(tasks) if t == T]
    if not idx: continue
    print(f"\n{'='*15} {T} ({len(idx)}건) {'='*15}")
    evaluate([gold[i] for i in idx], [pred[i] for i in idx])

print(f"\n{'='*15} 전체 ({len(gold)}건) {'='*15}")
evaluate(gold, pred)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

 98%|█████████▊| 44/45 [10:43<00:21, 21.86s/it]